In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, OrdinalEncoder, RobustScaler, PowerTransformer, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, SimpleImputer, IterativeImputer
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, HalvingGridSearchCV, cross_val_score, KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest, AdaBoostClassifier, StackingClassifier, ExtraTreesClassifier
from sklearn.metrics import auc, roc_curve, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone
from sklearn.feature_selection import SelectFromModel

from skopt import BayesSearchCV

from scipy.stats import mstats

from xgboost import XGBClassifier

from catboost import CatBoostClassifier, Pool

import optuna

import lightgbm as lgb

In [3]:
train = pd.read_csv("train.csv", index_col=0)
test = pd.read_csv("test.csv", index_col=0)

In [4]:
sklearn.set_config(transform_output="pandas")

In [5]:
train.head()

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
id,,,,,,,,,,,,,
0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [6]:
train['addicted_label'].value_counts()

addicted_label
1    490474
0    200895
Name: count, dtype: int64

In [7]:
train.isnull().sum() / train.count()

age                        0.043670
daily_screen_time_hours    0.160960
social_media_hours         0.240404
gaming_hours               0.224642
work_study_hours           0.080516
sleep_hours                0.068760
notifications_per_day      0.108345
app_opens_per_day          0.132169
weekend_screen_time        0.193444
gender                     0.043836
stress_level               0.086681
academic_work_impact       0.068337
addicted_label             0.000000
dtype: float64

In [8]:
(pd.get_dummies(train, dtype=int, drop_first=True).corr() > .3)

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,addicted_label,gender_Male,gender_Other,stress_level_Low,stress_level_Medium,academic_work_impact_Yes
age,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
daily_screen_time_hours,False,True,True,True,True,False,False,False,True,True,False,False,False,False,False
social_media_hours,False,True,True,False,False,False,False,False,True,True,False,False,False,False,False
gaming_hours,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False
work_study_hours,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False
sleep_hours,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
notifications_per_day,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
app_opens_per_day,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
weekend_screen_time,False,True,True,True,True,False,False,False,True,True,False,False,False,False,False
addicted_label,False,True,True,False,False,False,False,False,True,True,False,False,False,False,False


In [9]:
num_cols = train.loc[:, lambda x : x.dtypes == 'float64'].columns.to_list()
ord_cols = ['stress_level']
nom_cols = ['gender', 'academic_work_impact']
cat_cols = ['gender', 'academic_work_impact', 'stress_level']

In [10]:
from sklearn.model_selection import train_test_split

X, y = train.drop(labels="addicted_label", axis=1), train["addicted_label"]

train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=.3, random_state=42)

In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

simple_train = pd.get_dummies(train_X, dtype=int).corr() > .3

vif = pd.DataFrame()
vif["feature"] = simple_train.columns

vif["vif"] = [variance_inflation_factor(simple_train.values, i) for i in range(len(simple_train.columns))]

c:\Users\Ryan\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [12]:
vif

,feature,vif
0,age,1.000000
1,daily_screen_time_hours,inf
2,social_media_hours,2.161765
3,gaming_hours,2.161765
4,work_study_hours,2.161765
5,sleep_hours,1.000000
6,notifications_per_day,1.000000
7,app_opens_per_day,1.000000
8,weekend_screen_time,inf
9,gender_Female,1.000000


In [13]:
def categorical_handler(df):
    df_c = df.copy()

    for col in nom_cols:
        df_c[col] = df_c[col].astype(str).replace('nan', 'missing').astype('category')
    stress_map = {'Low':0, 'Medium':1, 'High':2}
    df_c['stress_level'] = df_c['stress_level'].map(stress_map).fillna(-1).astype(int)

    return df_c

In [14]:
categorical_handler(train_X)

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact
id,,,,,,,,,,,,
412708,28.0,9.49,3.67,1.23,3.76,8.26,128.0,180.0,13.21,Other,2,No
218301,35.0,NaN,0.79,NaN,1.50,4.63,224.0,78.0,NaN,Other,0,No
488140,NaN,11.57,2.69,1.76,2.52,8.04,68.0,155.0,10.16,Female,0,No
371198,18.0,9.17,2.98,2.98,0.81,7.34,123.0,151.0,6.07,Other,1,Yes
329533,20.0,5.55,1.76,2.51,NaN,5.27,243.0,42.0,NaN,Female,2,No
...,...,...,...,...,...,...,...,...,...,...,...,...
259178,NaN,12.96,4.94,0.71,5.26,6.70,87.0,36.0,13.29,missing,2,No
365838,NaN,11.54,4.86,NaN,4.43,8.42,22.0,45.0,15.83,missing,2,Yes
131932,33.0,8.23,4.26,0.33,2.93,5.71,245.0,62.0,14.10,Other,2,Yes


In [15]:
categorical_transformer = FunctionTransformer(categorical_handler, validate=False)

In [16]:
num_pipe = Pipeline(steps=[('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('outlier', PowerTransformer())])
nominal_pipe = Pipeline(steps=[('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
preprocessing = ColumnTransformer(transformers=[("num", num_pipe, num_cols),
                                                ("nom", nominal_pipe, nom_cols)],
                                                remainder='passthrough'
                                                 )

In [17]:
feature_selector = SelectFromModel(
    estimator=ExtraTreesClassifier(n_estimators=50, random_state=42),
    threshold='median'
)

In [18]:
def drop_features(estimator):
    feature_selector = SelectFromModel(
    estimator=estimator,
    threshold='median'
)
    p = Pipeline(steps=[('data_casting', categorical_transformer), ('preprocessing', preprocessing), ('drop', feature_selector)])
    p.fit(train_X, train_y)
    drop_step = p.named_steps['drop']
    return ColumnTransformer(transformers=[('keep', 'passthrough', drop_step.get_support().tolist())])

In [19]:
clf_pipe = Pipeline(steps=[('data_casting', categorical_transformer), ('preprocessing', preprocessing),('drop', 'passthrough'), ('model', 'passthrough')])

In [20]:
    
def CV_pruner(classifier, pipe, trial):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = [] 
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(train_X, train_y)):
        X_tr, X_val = train_X.iloc[train_idx], train_X.iloc[val_idx]
        y_tr, y_val = train_y.iloc[train_idx], train_y.iloc[val_idx]


        model = clone(classifier)
        temp_p = clone(pipe)
        temp_p.set_params(model=model)

        temp_p.fit(X_tr, y_tr)
    
        preds = temp_p.predict_proba(X_val)[:, 1]
        fold_score = roc_auc_score(y_val, preds)
        scores.append(fold_score)
            
        current_mean = np.mean(scores)
        trial.report(current_mean, step=fold)
            
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)

In [21]:
trial1_pipe = clone(clf_pipe)
pipe2_params = {
        'preprocessing__num__outlier__standardize':False
    }
trial1_pipe.set_params(**pipe2_params)

def objective1(trial):
    depth = trial.suggest_int('max_depth', 0, 50)
    params = {
        'max_depth':None if depth == 0 else depth,
        'max_leaf_nodes':trial.suggest_int('max_leaf_nodes', 2, 50),
        'min_samples_leaf':trial.suggest_int('min_samples_leaf', 2, 10),
    }
    classifier = HistGradientBoostingClassifier(random_state=42, categorical_features="from_dtype", early_stopping='auto', **params)

    return CV_pruner(classifier, trial1_pipe, trial)

In [22]:
study1 = optuna.create_study(
direction='maximize',
pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)

study1.optimize(objective1, n_jobs=-1, n_trials=50)

[I 2026-08-30 15:32:20,255] A new study created in memory with name: no-name-8427e985-d283-4efe-af18-fd2f5680357d
[I 2026-08-30 15:35:34,826] Trial 19 finished with value: 0.931359745692142 and parameters: {'max_depth': 37, 'max_leaf_nodes': 4, 'min_samples_leaf': 6}. Best is trial 19 with value: 0.931359745692142.
[I 2026-08-30 15:35:36,232] Trial 15 finished with value: 0.9290296089752934 and parameters: {'max_depth': 2, 'max_leaf_nodes': 6, 'min_samples_leaf': 9}. Best is trial 19 with value: 0.931359745692142.
[I 2026-08-30 15:35:45,201] Trial 4 finished with value: 0.9217323596067768 and parameters: {'max_depth': 30, 'max_leaf_nodes': 2, 'min_samples_leaf': 6}. Best is trial 19 with value: 0.931359745692142.
[I 2026-08-30 15:36:01,548] Trial 0 finished with value: 0.9361494723063701 and parameters: {'max_depth': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.9361494723063701.
[I 2026-08-30 15:36:03,458] Trial 7 finished with value: 0.9290296089752934

In [23]:
trial2_pipe = clone(clf_pipe)
pipe2_params = {
        'preprocessing':'passthrough',
    }
trial2_pipe.set_params(**pipe2_params)

def objective2(trial):
    params = {
        'max_depth':trial.suggest_int('max_depth', 0, 50),
        'learning_rate':trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'min_child_weight':trial.suggest_float('min_child_weight', 1, 10)
    }
    classifier = XGBClassifier(random_state=42, enable_categorical=True, n_jobs=-1, **params)

    return CV_pruner(classifier, trial2_pipe, trial)

In [24]:
study2 = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)
study2.optimize(objective2, n_trials=50)

[I 2026-08-30 15:47:42,996] A new study created in memory with name: no-name-23ee1b76-3be1-43a8-bbf2-56e7de708314


[I 2026-08-30 15:47:50,692] Trial 0 finished with value: 0.9539065892710529 and parameters: {'max_depth': 10, 'learning_rate': 0.0745510714779866, 'min_child_weight': 6.144630493107862}. Best is trial 0 with value: 0.9539065892710529.
[I 2026-08-30 15:47:58,522] Trial 1 finished with value: 0.9497380407727682 and parameters: {'max_depth': 10, 'learning_rate': 0.0538606141647778, 'min_child_weight': 4.4839940488973}. Best is trial 0 with value: 0.9539065892710529.
[I 2026-08-30 15:48:06,100] Trial 2 finished with value: 0.9596761092430771 and parameters: {'max_depth': 11, 'learning_rate': 0.16324307488309248, 'min_child_weight': 9.870922736335093}. Best is trial 2 with value: 0.9596761092430771.
[I 2026-08-30 15:49:19,592] Trial 3 finished with value: 0.9445090452423134 and parameters: {'max_depth': 27, 'learning_rate': 0.03347064153237455, 'min_child_weight': 2.637577847341617}. Best is trial 2 with value: 0.9596761092430771.
[I 2026-08-30 15:51:51,786] Trial 4 finished with value: 0.9

In [25]:
trial3_pipe = clone(clf_pipe)
pipe3_params = {
        'preprocessing':'passthrough',
    }
trial3_pipe.set_params(**pipe3_params)


def objective3(trial):
    params = {
        'max_depth':trial.suggest_int('max_depth', 4, 10),
        'learning_rate':trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_samples':trial.suggest_float('min_child_samples', 1, 10),
    }
    classifier = CatBoostClassifier(
        **params,
        cat_features=['gender', 'academic_work_impact'], 
        early_stopping_rounds=20,
        task_type='GPU',
        random_state=42,
        iterations=250,
        verbose=False
    )    

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = [] 
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(train_X, train_y)):
        X_tr, X_val = train_X.iloc[train_idx], train_X.iloc[val_idx]
        y_tr, y_val = train_y.iloc[train_idx], train_y.iloc[val_idx]


        model = clone(classifier)
        temp_p = clone(trial3_pipe)

        X_tr_proc = temp_p.fit_transform(X_tr, y_tr)
        X_val_proc = temp_p.transform(X_val)

        model.fit(X_tr_proc, y_tr, eval_set=(X_val_proc, y_val))
    
        preds = model.predict_proba(X_val_proc)[:, 1]
        fold_score = roc_auc_score(y_val, preds)
        scores.append(fold_score)
            
        current_mean = np.mean(scores)
        trial.report(current_mean, step=fold)
            
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)

In [26]:
study3 = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)
study3.optimize(objective3, n_trials=50)

[I 2026-08-30 16:01:35,598] A new study created in memory with name: no-name-6364c3f1-e3ea-431c-9fe8-30b8716aae96
[I 2026-08-30 16:02:06,947] Trial 0 finished with value: 0.9459502024157744 and parameters: {'max_depth': 7, 'learning_rate': 0.050045091469182484, 'min_child_samples': 1.667362756301001}. Best is trial 0 with value: 0.9459502024157744.
[I 2026-08-30 16:03:27,918] Trial 1 finished with value: 0.9517991396949796 and parameters: {'max_depth': 10, 'learning_rate': 0.06137488573325068, 'min_child_samples': 2.1213468292373605}. Best is trial 1 with value: 0.9517991396949796.
[I 2026-08-30 16:03:54,111] Trial 2 finished with value: 0.9366414237121802 and parameters: {'max_depth': 6, 'learning_rate': 0.022198397314791417, 'min_child_samples': 9.2237759624896}. Best is trial 1 with value: 0.9517991396949796.
[I 2026-08-30 16:04:25,528] Trial 3 finished with value: 0.9441252122776577 and parameters: {'max_depth': 7, 'learning_rate': 0.04116416116565907, 'min_child_samples': 5.170188

In [27]:
trial4_pipe = clone(clf_pipe)
pipe4_params = {
        'preprocessing__num__outlier__standardize':False,
        'drop':drop_features(ExtraTreesClassifier(random_state=42, n_estimators=50))
    }
trial4_pipe.set_params(**pipe4_params)

def objective4(trial):
    depth = trial.suggest_int('max_depth', 0, 50)
    params = {
        'max_depth':None if depth == 0 else depth,
        'max_leaf_nodes':trial.suggest_int('max_leaf_nodes', 2, 50),
        'min_samples_split':trial.suggest_int('min_samples_split', 5, 10),
        'min_samples_leaf':trial.suggest_int('min_samples_leaf', 2, 10),
    }
    classifier = RandomForestClassifier(**params, n_jobs=-1, random_state=42)

    return CV_pruner(classifier, trial4_pipe, trial)

In [28]:
study4 = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)
study4.optimize(objective4, n_trials=50)

[I 2026-08-30 16:28:42,950] A new study created in memory with name: no-name-a7f6ee73-ebe9-4ee8-9998-ad711cf40382
[I 2026-08-30 16:29:23,121] Trial 0 finished with value: 0.9165977869303003 and parameters: {'max_depth': 37, 'max_leaf_nodes': 12, 'min_samples_split': 5, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.9165977869303003.
[I 2026-08-30 16:30:06,509] Trial 1 finished with value: 0.9260871708039483 and parameters: {'max_depth': 25, 'max_leaf_nodes': 39, 'min_samples_split': 5, 'min_samples_leaf': 10}. Best is trial 1 with value: 0.9260871708039483.
[I 2026-08-30 16:30:49,454] Trial 2 finished with value: 0.9243599383599852 and parameters: {'max_depth': 38, 'max_leaf_nodes': 30, 'min_samples_split': 10, 'min_samples_leaf': 9}. Best is trial 1 with value: 0.9260871708039483.
[I 2026-08-30 16:31:32,526] Trial 3 finished with value: 0.9243599383599852 and parameters: {'max_depth': 46, 'max_leaf_nodes': 30, 'min_samples_split': 10, 'min_samples_leaf': 10}. Best is trial 1 wi

In [29]:
trial5_pipe = clone(clf_pipe)
pipe5_params = {
        'drop':drop_features(LogisticRegression(random_state=42))
    }
trial5_pipe.set_params(**pipe5_params)

def objective5(trial):
    params = {
        'C':trial.suggest_float('C', .0001, 100, log=True),
        'l1_ratio':trial.suggest_float('l1_ratio', 0, 1, step=.01),
    }
    classifier = LogisticRegression(random_state=42, **params, solver='saga')

    return CV_pruner(classifier, trial5_pipe, trial)

In [30]:
study5 = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)
study5.optimize(objective5, n_jobs=-1, n_trials=50)

[I 2026-08-30 16:57:31,032] A new study created in memory with name: no-name-6e2eba17-1927-4aa7-9cdf-aaeee387f36b
c:\Users\Ryan\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\Ryan\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\Ryan\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\Ryan\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\Ryan\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only 

In [31]:
from sklearn.calibration import CalibratedClassifierCV

trial6_pipe = clone(clf_pipe)
pipe6_params = {
        'drop':drop_features(LogisticRegression(random_state=42))
    }
trial6_pipe.set_params(**pipe6_params)

def objective6(trial):
    params = {
        'penalty':trial.suggest_categorical('penalty', ['l1', 'l2']),
        'alpha':trial.suggest_float('alpha', 1e-6, 100, log=True),
        'tol':trial.suggest_float('tol', 1e-4, 1e-2, log=True),
    }
    classifier = CalibratedClassifierCV(SGDClassifier(**params, loss='hinge', random_state=42))

    return CV_pruner(classifier, trial6_pipe, trial)

In [32]:
study6 = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)
study6.optimize(objective6, n_jobs=-1, n_trials=50)

[I 2026-08-30 17:06:33,744] A new study created in memory with name: no-name-76a38cf9-12fa-4169-a40d-8261429f2412
[I 2026-08-30 17:10:09,297] Trial 7 finished with value: 0.5 and parameters: {'penalty': 'l1', 'alpha': 0.5489723105066525, 'tol': 0.0006716466329145957}. Best is trial 7 with value: 0.5.
[I 2026-08-30 17:10:11,253] Trial 10 finished with value: 0.906757420875571 and parameters: {'penalty': 'l2', 'alpha': 0.6565722843020535, 'tol': 0.0003990722261810334}. Best is trial 10 with value: 0.906757420875571.
[I 2026-08-30 17:10:13,350] Trial 18 finished with value: 0.9113790134691296 and parameters: {'penalty': 'l1', 'alpha': 0.05556747236505547, 'tol': 0.0005525201595083202}. Best is trial 18 with value: 0.9113790134691296.
[I 2026-08-30 17:10:15,756] Trial 0 finished with value: 0.9120830614996173 and parameters: {'penalty': 'l1', 'alpha': 0.016340725454432166, 'tol': 0.003120221286754054}. Best is trial 0 with value: 0.9120830614996173.
[I 2026-08-30 17:10:16,080] Trial 2 fini

In [33]:
trial7_pipe = clone(clf_pipe)
pipe7_params = {
        'drop':drop_features(ExtraTreesClassifier(random_state=42, n_estimators=50))
    }
trial7_pipe.set_params(**pipe7_params)

def objective7(trial):
    params = {
        'hidden_layer_sizes':(trial.suggest_int('hidden_layer_sizes', 1, 200),),
        'activation':trial.suggest_categorical('activation', ['identity', 'logistic', 'tanh', 'relu']),
        'alpha':trial.suggest_float('alpha', .0001, 100, log=True),
    }
    classifier = MLPClassifier(**params, early_stopping=True, random_state=42)

    return CV_pruner(classifier, trial7_pipe, trial)

In [34]:
study7 = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)
study7.optimize(objective7, n_jobs=-1, n_trials=50)

[I 2026-08-30 17:16:44,541] A new study created in memory with name: no-name-71c054a2-a807-4e8c-b706-92e052f674ec
[I 2026-08-30 17:25:03,964] Trial 5 finished with value: 0.9045260216972588 and parameters: {'hidden_layer_sizes': 35, 'activation': 'relu', 'alpha': 74.30432777279925}. Best is trial 5 with value: 0.9045260216972588.
[I 2026-08-30 17:29:39,820] Trial 3 finished with value: 0.9167465921213095 and parameters: {'hidden_layer_sizes': 12, 'activation': 'logistic', 'alpha': 1.1852037758718754}. Best is trial 3 with value: 0.9167465921213095.
[I 2026-08-30 17:33:04,369] Trial 19 finished with value: 0.9033587030438632 and parameters: {'hidden_layer_sizes': 179, 'activation': 'logistic', 'alpha': 26.67817665227272}. Best is trial 3 with value: 0.9167465921213095.
[I 2026-08-30 17:37:56,349] Trial 13 finished with value: 0.9287451396672953 and parameters: {'hidden_layer_sizes': 26, 'activation': 'tanh', 'alpha': 0.6632374751492083}. Best is trial 13 with value: 0.9287451396672953.


In [35]:
trial8_pipe = clone(clf_pipe)
pipe8_params = {
        'preprocessing':'passthrough',
        'drop':'passthrough'
    }
trial8_pipe.set_params(**pipe8_params)

def objective8(trial):
    params = {
        'max_depth':trial.suggest_int('max_depth', 1, 15),
        'learning_rate':trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':trial.suggest_int('num_leaves', 10, 60)
    }
    classifier = lgb.LGBMClassifier(random_state=42, objective='binary', **params)
    
    return CV_pruner(classifier, trial8_pipe, trial)

In [36]:
study8 = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)
study8.optimize(objective8, n_jobs=-1, n_trials=50)

[I 2026-08-30 18:57:10,990] A new study created in memory with name: no-name-1c093848-6796-45ee-9e58-60d39afbd9f3
[I 2026-08-30 18:57:25,102] Trial 12 finished with value: 0.9124500218587114 and parameters: {'max_depth': 1, 'learning_rate': 0.01944347757565657, 'num_leaves': 46}. Best is trial 12 with value: 0.9124500218587114.
[I 2026-08-30 18:57:30,548] Trial 6 finished with value: 0.9209421003603703 and parameters: {'max_depth': 2, 'learning_rate': 0.013615923830404724, 'num_leaves': 22}. Best is trial 6 with value: 0.9209421003603703.
[I 2026-08-30 18:57:51,609] Trial 11 finished with value: 0.9396667988896382 and parameters: {'max_depth': 15, 'learning_rate': 0.06291179908751945, 'num_leaves': 12}. Best is trial 11 with value: 0.9396667988896382.
[I 2026-08-30 18:57:54,075] Trial 19 finished with value: 0.9312772687222889 and parameters: {'max_depth': 4, 'learning_rate': 0.03959592000900278, 'num_leaves': 44}. Best is trial 11 with value: 0.9396667988896382.
[I 2026-08-30 18:58:14

In [38]:
estimators = [
        ('HGBC',trial1_pipe.set_params(model=HistGradientBoostingClassifier(random_state=42, **study1.best_params))),
        ('XGBC',trial2_pipe.set_params(model=XGBClassifier(random_state=42 ,enable_categorical=True, **study2.best_params))),
        ('CatBC',trial3_pipe.set_params(model=CatBoostClassifier(random_state=42,cat_features=nom_cols, early_stopping_rounds=20, verbose=False, **study3.best_params))),
        ('RF',trial4_pipe.set_params(model=RandomForestClassifier(random_state=42, **study4.best_params))),
        ('Log',trial5_pipe.set_params(model=LogisticRegression(random_state=42, solver='saga', **study5.best_params))),
        ('SVC',trial6_pipe.set_params(model=SGDClassifier(random_state=42, loss='hinge', **study6.best_params))),
        ('MLPCLass',trial7_pipe.set_params(model=MLPClassifier(random_state=42,**study7.best_params))),
        ('light',trial8_pipe.set_params(model=lgb.LGBMClassifier(random_state=42, objective='binary', **study8.best_params))),
]



clf = StackingClassifier(estimators, n_jobs=-1)

In [39]:
clf.fit(train_X, train_y)

StackingClassifier(estimators=[('HGBC',
                                Pipeline(steps=[('data_casting',
                                                 FunctionTransformer(func=<function categorical_handler at 0x0000020F670F9300>)),
                                                ('preprocessing',
                                                 ColumnTransformer(remainder='passthrough',
                                                                   transformers=[('num',
                                                                                  Pipeline(steps=[('imputer',
                                                                                                   SimpleImputer(add_indicator=True,
                                                                                                                 strategy='median')),
                                                                                                  ('outlier',
                                                                                                   PowerTransformer(standardize=Fa...
                                                               hidden_layer_sizes=198,
                                                               random_state=42))])),
                               ('light',
                                Pipeline(steps=[('data_casting',
                                                 FunctionTransformer(func=<function categorical_handler at 0x0000020F670F9300>)),
                                                ('preprocessing',
                                                 'passthrough'),
                                                ('drop', 'passthrough'),
                                                ('model',
                                                 LGBMClassifier(learning_rate=0.18198150559137785,
                                                                max_depth=11,
                                                                num_leaves=59,
                                                                objective='binary',
                                                                random_state=42))]))],
                   n_jobs=-1)

In [40]:
test_clf = clf.predict(test_X)

c:\Users\Ryan\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [41]:
roc_auc_score(y_true=test_y, y_score=test_clf)

0.8741344340989202

In [42]:
clf.score(test_X, test_y)

c:\Users\Ryan\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


0.9009551084561571

In [43]:
outf = pd.DataFrame({'id':test.index, 'addicted_label':clf.predict_proba(test)[:, 1]})

c:\Users\Ryan\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [44]:
outf.to_csv('AugustSubmission.csv', index=False)